# ML Pipeline — From Raw Data to Predictions

**The complete Machine Learning pipeline in one notebook.**

Loads call center data → cleans → engineers features → trains models → evaluates → explains with SHAP.

No cloud. No MLflow. Just Python + scikit-learn. This is where every ML project starts.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# On Colab: clone the repo to get the data
if not os.path.exists("data") and not os.path.exists("../../data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/repo 2>/dev/null
    DATA_DIR = "/tmp/repo/data"
elif os.path.exists("../../data/calls.json"):
    DATA_DIR = "../../data"
elif os.path.exists("data/calls.json"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.expanduser("~/Downloads/systems-in-production/data")

print(f"Data directory: {DATA_DIR}")

---

## Stage 1: BRONZE — Load Raw Data

Same raw data as the Analytics Pipeline. Both pipelines start from the same source.

In [ ]:
calls_raw = pd.read_json(os.path.join(DATA_DIR, "calls.json"), lines=True)
campaigns_raw = pd.read_csv(os.path.join(DATA_DIR, "campaigns.csv"))
orders_raw = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))

print(f"calls:     {len(calls_raw):,} records")
print(f"campaigns: {len(campaigns_raw):,} records")
print(f"orders:    {len(orders_raw):,} records")

---

## Stage 2: SILVER — Clean the Data

In [ ]:
calls = calls_raw.copy()

# --- Flatten nested customer data ---
# WHY: The raw JSON has a nested "customer" object inside each call record:
#   {"call_id": "C001", "duration": 120, "customer": {"first_name": "Jane", "last_name": "Doe", "address": {"city": "Austin", "state": "TX"}}}
#
# Pandas can't use nested objects as features. We need flat columns.
# json_normalize() turns the nested structure into flat columns:
#   customer_first_name, customer_last_name, customer_address_city, customer_address_state
#
# This is a common real-world problem — APIs and NoSQL databases (MongoDB)
# return nested JSON. SQL databases and ML models need flat tables.
# The DE pipeline does this same flattening in the Silver layer.

if "customer" in calls.columns:
    customer_df = pd.json_normalize(calls["customer"])
    customer_df.columns = ["customer_" + c.replace(".", "_") for c in customer_df.columns]
    calls = pd.concat([calls.drop(columns=["customer"]), customer_df], axis=1)
    print(f"Flattened nested 'customer' object into {len(customer_df.columns)} columns:")
    print(f"  {list(customer_df.columns)}")

# --- Deduplicate ---
# WHY: Source systems sometimes send the same call twice (network retry,
# webhook fired twice, import ran twice). Duplicates inflate counts
# and cause the model to train on the same call twice — biasing the results.
# We keep the first occurrence and drop the rest.

before = len(calls)
calls = calls.drop_duplicates(subset=["call_id"], keep="first")
print(f"\nDuplicates removed: {before - len(calls)} (from {before} → {len(calls)})")

# --- Fix data types ---
# WHY: Dates arrive as strings ("2026-03-17"). Pandas needs them as datetime
# objects to extract hour, day_of_week, etc. for feature engineering.
# DNIS arrives as a number (8005551001) but we treat it as a string for joining
# (it's an identifier, not a number to do math on).
# Disposition and channel have inconsistent casing ("completed" vs "Completed")
# — standardize to lowercase/uppercase so groupby works correctly.

calls["date"] = pd.to_datetime(calls["date"], errors="coerce")
calls["start_time"] = pd.to_datetime(calls["start_time"], errors="coerce", utc=True)
calls["end_time"] = pd.to_datetime(calls["end_time"], errors="coerce", utc=True)
calls["dnis"] = calls["dnis"].astype(str)
calls["disposition"] = calls["disposition"].str.strip().str.lower()
calls["channel"] = calls["channel"].str.strip().str.upper()

print(f"\nSilver calls: {len(calls):,} records")
print(f"Date range: {calls['date'].min()} to {calls['date'].max()}")
print(f"Dispositions: {dict(calls['disposition'].value_counts())}")

---

## Stage 3: GOLD (ML) — Feature Engineering

### The Use Case: Predicting Churn Risk

**What we are predicting:** For each call, will this caller become a **churn risk** — someone likely to stop using the service?

**Why it matters:** A call center handling 100 calls/day loses customers silently. If the company could identify which callers are at risk of leaving DURING or AFTER the call, they could intervene — offer a discount, transfer to a senior agent, send a follow-up email. Every saved customer is revenue retained.

**How we define "churn risk" in this data:** Since we do not have actual CRM (Customer Relationship Management) data showing who left, we **simulate** the label from call outcomes. A call that ended as `callback` (unresolved — they have to call again), `abandoned` (gave up waiting), or `voicemail` (no one answered) is labeled as churn risk. A `completed` call is labeled as no risk.

> **This is a simplification.** In production, churn would come from actual customer behavior data — did they cancel? Did they stop buying? Did they switch to a competitor? The simulated label is good enough to demonstrate the ML pipeline, but it is not good enough for real business decisions. That gap is intentional — it teaches that the target variable matters as much as the features.

**Key difference from Analytics Gold:** The ML Gold **drops records with missing features.** The analytics Gold keeps them all. Same Silver layer, different Gold consumers. The DE team keeps every record for accurate counts ("how many total calls?"). The ML team needs clean feature vectors — a missing value in `duration` would cause the model to crash or learn incorrect patterns.

In [ ]:
ml_data = calls.copy()

# --- Join campaign info ---
# WHY: The raw call record only has "dnis" (a phone number like 8005551001).
# A phone number is not a feature — the model can't learn from it.
# But the campaign TABLE tells us: which client, which campaign, which channel (VA vs Live).
# These are predictive: VA calls may have different churn patterns than live agent calls.
# Different campaigns attract different customer types.
# This is the same concept as joining dimension tables in the star schema —
# dimensions add CONTEXT to facts. Here, the context becomes ML features.

campaigns_raw["dnis"] = campaigns_raw["dnis"].astype(str)
ml_data = ml_data.merge(
    campaigns_raw[["dnis", "campaign_name", "client_name", "channel"]].rename(
        columns={"channel": "campaign_channel"}),
    on="dnis", how="left"
)
print(f"After campaign join: {len(ml_data)} rows, added: campaign_name, client_name, campaign_channel")

# --- Join order info (did the caller place an order?) ---
# WHY: The call record doesn't say whether the caller bought anything.
# That information lives in the ORDERS table — a separate system.
# By joining, we create features the model can actually use:
#   - placed_order (yes/no): a caller who bought is less likely to churn
#   - total_revenue: higher spend may correlate with satisfaction
#   - order_count: multiple items may signal engagement
# Without this join, the model has no purchase behavior signal at all.

order_summary = orders_raw.groupby("call_id").agg(
    order_count=("order_id", "count"),
    total_revenue=("total", "sum")
).reset_index()
ml_data = ml_data.merge(order_summary, on="call_id", how="left")
ml_data["order_count"] = ml_data["order_count"].fillna(0).astype(int)
ml_data["total_revenue"] = ml_data["total_revenue"].fillna(0)
ml_data["placed_order"] = (ml_data["order_count"] > 0).astype(int)

print(f"After order join: added: placed_order, order_count, total_revenue")
print(f"Callers who placed an order: {ml_data['placed_order'].sum()} / {len(ml_data)} ({ml_data['placed_order'].mean():.1%})")
print(f"\nAll columns now: {list(ml_data.columns)}")

In [ ]:
# --- Traditional features (from existing columns) ---
ml_data["hour_of_day"] = ml_data["start_time"].dt.hour
ml_data["is_weekend"] = (ml_data["start_time"].dt.dayofweek >= 5).astype(int)
ml_data["is_va"] = (ml_data["campaign_channel"] == "VA").astype(int)

# --- Target variable ---
# Churn risk proxy: callback, abandoned, or voicemail = unresolved = churn risk
# In production, actual churn would come from CRM data.
ml_data["is_churn_risk"] = ml_data["disposition"].isin(
    ["callback", "abandoned", "voicemail"]
).astype(int)

print(f"Churn risk rate: {ml_data['is_churn_risk'].mean():.1%}")
print(f"\nDisposition breakdown:")
ml_data["disposition"].value_counts()

In [ ]:
# --- Drop nulls in critical features ---
# Analytics keeps these rows. ML drops them.
critical = ["duration", "hour_of_day"]
before = len(ml_data)
ml_data = ml_data.dropna(subset=critical)
dropped = before - len(ml_data)
print(f"Dropped {dropped} records with missing features")
print(f"ML-ready records: {len(ml_data):,}")

---

## Stage 4: Model Training and Evaluation

### The Two Algorithms — and Why These Two

**Logistic Regression** — the simplest classification algorithm. Despite the name, it is a classifier, not a regression. It draws a straight line (or flat surface) through the feature space to separate the two classes (churn risk vs no risk). For each call, it computes a weighted sum of the features, passes it through a sigmoid function (squashes any number into 0-1 range), and outputs a probability of churn risk.

**Analogy:** Like a scorecard. Each feature gets a weight: duration × 0.3 + hour × 0.1 + is_va × 0.5 + ... = total score. If the score is above a threshold, predict "churn risk."

**Why we use it:** It is fast, explainable (each feature gets a clear weight), and serves as a **baseline** — if a complex model can't beat logistic regression, the features have no signal.

---

**Random Forest** — an ensemble of decision trees. Each tree asks a series of yes/no questions about the features ("is duration > 120? is_va = 1?") and reaches a prediction. A forest of 100 trees each votes, and the majority vote wins. Each tree is trained on a random subset of the data and features, which reduces overfitting.

**Analogy:** Like asking 100 different experts (each trained on slightly different data) and going with the majority opinion. One expert might be wrong, but 100 experts together are usually right.

**Why we use it:** It handles non-linear relationships (logistic regression can only draw straight lines), rarely needs feature scaling, and is the go-to algorithm for tabular data in production.

---

### Why These Two Together?

| | Logistic Regression | Random Forest |
|:---|:---|:---|
| **Boundary shape** | Straight line | Complex, non-linear |
| **Interpretability** | High — each feature has one weight | Medium — feature importance but no single formula |
| **Speed** | Very fast | Slower (100 trees) |
| **Role in this pipeline** | Baseline — "can a simple model find signal?" | Power model — "can a complex model find more?" |

If both perform poorly (as they do here), the problem is the features, not the algorithms. That is the diagnostic insight.

---

### The Metrics — What Each One Measures

In our case, the **two classes** are: **churn risk (1)** and **no risk (0)**. Every metric below measures how well the model separates these two groups.

**Accuracy** = correct predictions / total predictions. Misleading when classes are imbalanced — predicting "no risk" for everyone gets 85% accuracy but catches zero actual risks.

**Precision** = true positives / (true positives + false positives). "Of everyone I flagged as churn risk, how many actually are?" High precision = fewer false alarms.

**Recall** = true positives / (true positives + false negatives). "Of everyone who actually IS a churn risk, how many did I catch?" High recall = fewer missed cases.

**F1 Score** = the harmonic mean of precision and recall. Balances both in one number.

```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

Why harmonic mean? If precision is 99% but recall is 1%, a regular average says 50% (looks OK). The harmonic mean says 2% (correctly flags the disaster).

---

**ROC-AUC (Receiver Operating Characteristic — Area Under Curve, pronounced "rock awk")**

The model does not directly output "churn risk" or "no risk." It outputs a **probability** — a number between 0 and 1. For example: 0.73 (73% chance of churn risk). We then apply a **threshold** to convert the probability into a yes/no decision.

- Threshold = 0.5 (default): probability ≥ 0.5 → predict "churn risk." Below → predict "no risk."
- Threshold = 0.3 (lower): catches more churn risks (higher recall) but also more false alarms (lower precision)
- Threshold = 0.8 (higher): only flags very confident predictions (higher precision) but misses borderline cases (lower recall)

**The problem:** Precision, recall, and F1 all depend on which threshold we choose. Change the threshold, the numbers change.

**ROC-AUC solves this** by evaluating the model across ALL possible thresholds at once. It asks: "If I pick a random churn risk caller and a random no-risk caller, what is the probability that the model assigns a HIGHER probability to the churn risk caller?"

```
AUC = 1.0 → the model ALWAYS ranks churn risks higher than non-risks (perfect)
AUC = 0.5 → the model ranks randomly — no better than flipping a coin (useless)
AUC = 0.7 → 70% of the time, the model correctly ranks a churn risk higher
```

**Analogy:** Imagine sorting a mixed pile of red balls (churn risk) and blue balls (no risk) by "how red they look." AUC = 1.0 means all reds ended up on one side and all blues on the other. AUC = 0.5 means the sort was random — reds and blues are mixed evenly.

---

**5-Fold Cross-Validation** = instead of one train/test split (which might be lucky or unlucky), split the data 5 ways. Train 5 times, each time a different 20% is the test set. Report the average and the spread (±).

```
Fold 1: [TEST] [Train] [Train] [Train] [Train] → F1: 0.24
Fold 2: [Train] [TEST] [Train] [Train] [Train] → F1: 0.21
Fold 3: [Train] [Train] [TEST] [Train] [Train] → F1: 0.27
Fold 4: [Train] [Train] [Train] [TEST] [Train] → F1: 0.22
Fold 5: [Train] [Train] [Train] [Train] [TEST] → F1: 0.25

Average: 0.238 ± 0.023  (reliable — consistent across splits)
```

A wide spread (e.g., 0.05, 0.40, 0.10) means the model is unstable — good on some splits, bad on others. A tight spread means the result is trustworthy.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler

# Select features
feature_cols = ["duration", "hour_of_day", "is_weekend", "is_va",
                "placed_order", "order_count", "total_revenue"]

X = ml_data[feature_cols].fillna(0)
y = ml_data["is_churn_risk"]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Target: is_churn_risk")
print(f"Class balance: {dict(y.value_counts())}")

In [ ]:
# Split (stratify keeps the class ratio the same in train and test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (Logistic Regression needs this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# Train and evaluate both models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
}

results = []

for name, model in models.items():
    if "Logistic" in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="f1")
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if len(y_test.unique()) > 1 else 0

    results.append({"Model": name, "Accuracy": acc, "Precision": prec,
                     "Recall": rec, "F1": f1, "ROC-AUC": auc})

    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (of flagged churn risks, how many are real?)")
    print(f"  Recall:    {rec:.3f}  (of actual risks, how many caught?)")
    print(f"  F1:        {f1:.3f}")
    print(f"  ROC-AUC:   {auc:.3f}")
    print(f"  5-Fold CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# Model comparison table
results_df = pd.DataFrame(results).set_index("Model")
print("MODEL COMPARISON")
results_df.round(3)

### The Honest Assessment

Both models perform poorly. This is **not a bug — it is the lesson.**

The features available (duration, hour, weekend, channel, order info) do not predict churn risk. Knowing a call happened at 3 PM on a weekday via the VA channel tells the model almost nothing about whether the customer will churn.

**What WOULD predict churn?**
- The call transcript ("I'm frustrated, this is the third time I've called")
- Customer sentiment (angry → churn risk)
- Call history (5 calls in 2 weeks = unresolved problem)
- The reason for the call (billing dispute vs product question)

These are **AI-derived features** — they require an LLM (Large Language Model) to analyze the transcript and extract the signal. No spreadsheet column captures "frustrated customer who mentioned a competitor."

**The principle: bad features produce bad models, regardless of the algorithm.** No amount of hyperparameter tuning fixes missing signal. The next step is not a better model — it is better features.

---

## Stage 5: Explainability — SHAP

Even a weak model can be explained. SHAP (SHapley Additive exPlanations, pronounced "shap") decomposes every prediction into per-feature contributions.

### What SHAP Actually Calculates

The model predicts "12% churn risk" for a specific call. SHAP answers: **how much did each feature contribute to that 12%?**

It works by computing, for each feature: "what would the prediction be WITH this feature versus WITHOUT it?" — averaged across all possible combinations of the other features.

```
Base prediction (average across all calls):     15% churn risk
+ duration = 71 seconds:                        -5%  (shorter calls = less risk)
+ hour = 19 (7 PM):                             -3%  (evening = normal hours)
+ placed_order = yes:                            -4%  (buyers don't churn)
+ total_revenue = $80:                           -2%  (spent money = satisfied)
+ is_va = no (live agent):                       -0.2% (barely matters)
+ is_weekend = no:                               0%   (doesn't matter at all)
= Final prediction:                              0.8% churn risk
```

Each feature gets a positive or negative contribution. They all add up to the final prediction. This is not a rough estimate — it is mathematically exact, based on **Shapley values** from game theory (1953 Nobel Prize in Economics). The idea: fairly distribute the "credit" for the prediction among all features, accounting for interactions between them.

### How to Read the Feature Importance Chart

The bar chart shows **mean absolute SHAP value** per feature — how much each feature influences predictions across all test calls, on average.

- **Duration dominates.** Longer calls tend to be completed sales. Very short calls tend to be abandoned or hangups. Duration has signal — but it is a **lagging indicator** (you only know the duration after the call ends, so you cannot use it to intervene during the call).
- **Hour of day** is a distant second. Some hours have higher churn rates — possibly late-night calls with fewer agents available.
- **is_weekend = 0.0000** — literally zero importance. Weekend vs weekday makes no difference in this data.
- **Order-related features** (placed_order, order_count, total_revenue) each contribute a small amount. If someone bought something, they are less likely to be a churn risk.

### How to Read the Individual Prediction

For a single call, SHAP shows WHY the model predicted what it did:

- **Negative SHAP value (↓ risk):** This feature pushed the prediction AWAY from churn risk
- **Positive SHAP value (↑ risk):** This feature pushed the prediction TOWARD churn risk
- **The magnitude** shows how much — a SHAP of -0.19 is a strong push, -0.005 is barely noticeable

**Example interpretation:** A caller at 7 PM (normal hour), duration 71 seconds (reasonable), placed an order ($80 revenue), via live agent. Every feature pushes in the same direction — away from risk. The model predicts 0% churn probability. SHAP shows exactly why: this person got what they called for.

Now imagine the opposite: a call at 2 AM, duration 15 seconds, no order, VA channel. Every arrow would flip — all pointing toward risk. Same model, same features, completely different story. SHAP makes that story visible.

In [ ]:
try:
    import shap

    rf_model = models["Random Forest"]
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test)

    # Handle different SHAP output formats
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    elif shap_values.ndim == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values

    # Feature importance
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Mean |SHAP|": np.abs(shap_vals).mean(axis=0)
    }).sort_values("Mean |SHAP|", ascending=False)

    print("Feature Importance (SHAP — what the model relies on):")
    print("=" * 50)
    max_val = importance["Mean |SHAP|"].max()
    for _, row in importance.iterrows():
        bar_len = int(30 * row["Mean |SHAP|"] / max_val) if max_val > 0 else 0
        bar = "█" * bar_len
        print(f"  {row['Feature']:20s} {row['Mean |SHAP|']:.4f} {bar}")

except ImportError:
    print("SHAP not installed. Run: pip install shap")

In [ ]:
# Explain one specific prediction
try:
    idx = 0
    pred = rf_model.predict(X_test.iloc[[idx]])[0]
    proba = rf_model.predict_proba(X_test.iloc[[idx]])[0][1]
    actual = y_test.iloc[idx]

    print(f"Example prediction (test row {idx}):")
    print(f"  Predicted: {'CHURN RISK' if pred == 1 else 'NO RISK'} ({proba:.1%} probability)")
    print(f"  Actual:    {'CHURN RISK' if actual == 1 else 'NO RISK'}")
    print(f"  Why:")
    for feat, val, sv in zip(feature_cols, X_test.iloc[idx], shap_vals[idx]):
        direction = "↑ risk" if sv > 0 else "↓ risk"
        print(f"    {feat:20s} = {val:8.1f}  →  SHAP: {sv:+.4f} ({direction})")
except:
    print("SHAP not available for individual explanation.")

---

## Summary and Next Steps

In [ ]:
print("ML PIPELINE COMPLETE")
print("=" * 50)
print(f"Bronze: {len(calls_raw):,} raw records")
print(f"Silver: {len(calls):,} cleaned records")
print(f"Gold (ML): {len(ml_data):,} feature-ready records")
try:
    print(f"Best model: {results_df['F1'].idxmax()} (F1: {results_df['F1'].max():.3f})")
except NameError:
    print("Best model: (run the model training cells above to see results)")
print()
print("KEY LESSON: The models are weak because the features are weak.")
print("Duration and hour_of_day don't predict churn. The transcript does.")
print()
print("NEXT STEPS:")
print("  1. AI-derived features: classify transcripts with an LLM")
print("     (disposition reason, sentiment, customer intent)")
print("  2. Better target: actual churn from CRM, not simulated from dispositions")
print("  3. More data: 500 calls is small. 50,000 would give the model more to learn.")
print("  4. Production: MLflow for tracking, FastAPI for serving, monitoring for drift")